# Projeto AWS: O Peso das Palavras
---

## Problema de Negócio
Em marketplaces digitais, nem sempre os clientes atribuem notas coerentes com seus comentários, ou até mesmo deixam de avaliar numericamente um produto. Dessa forma, surge a necessidade de desenvolver modelos que consigam interpretar automaticamente o sentimento do texto e convertê-lo em uma nota de satisfação.

## Objetivo
Construir um modelo preditivo capaz de estimar a nota de um produto (de 1 a 5) com base no conteúdo textual dos comentários.
O Desafio Técnico: utilizar um modelo de Regressão Linear para encontrar a relação entre o "tom" do texto e a "nota" atribuída.

In [ ]:
# !pip install pandas tqdm matplotlib seaborn nltk leia-br transformers torch tiktoken

In [2]:
# Gerais
import re
import pandas as pd
import os
from tqdm import tqdm
tqdm.pandas()
import warnings
warnings.filterwarnings('ignore')

# Gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Análise de Sentimento
import nltk
from nltk.corpus import stopwords
from LeIA import SentimentIntensityAnalyzer as LEIAAnalyzer
from nltk.sentiment import SentimentIntensityAnalyzer as VaderAnalyzer
from transformers import pipeline
from tqdm import tqdm
import torch

In [2]:
nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Giovanna\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Giovanna\AppData\Roaming\nltk_data...


True

## 1. Leitura das tabelas

Nesta etapa carregamos a tabela do dataset que será utilizada no projeto. 
O foco principal da análise está na tabela de **avaliações**, que contém informações textuais escritas pelos consumidores, além das notas atribuídas aos produtos.

A leitura inicial dos dados permite compreender:

- estrutura das tabelas
- quantidade de registros
- variáveis disponíveis
- valores nulos e duplicatas

In [3]:
import pandas as pd
import os

path = './bronze'

dfs = {}

for file in os.listdir(path):
  if file.endswith('.csv'):
    df_name = "df_" + file.replace(".csv", "")
    full_path = os.path.join(path, file)

    dfs[df_name] = pd.read_csv(full_path)

In [4]:
resultado = []

for nome, df in dfs.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    duplicatas = df.duplicated().sum()

    # Caso não tenha nulos
    if nulos.empty:
        resultado.append({
            "dataset": nome,
            "shape": df.shape,
            "coluna": None,
            "nulos": 0,
            "pct_nulos": 0.0,
            "duplicatas": duplicatas
        })
    else:
        for col, qtd in nulos.items():
            resultado.append({
                "dataset": nome,
                "shape": df.shape,
                "coluna": col,
                "nulos": qtd,
                "pct_nulos": round((qtd / len(df)) * 100, 1),
                "duplicatas": duplicatas
            })

df_resumo = pd.DataFrame(resultado)
display(df_resumo)

,dataset,shape,coluna,nulos,pct_nulos,duplicatas
0,df_avaliacoes,"(99224, 7)",review_comment_title,87656,88.3,0
1,df_avaliacoes,"(99224, 7)",review_comment_message,58247,58.7,0
2,df_clientes,"(99441, 5)",None,0,0.0,0
3,df_geolocalizacao,"(1000163, 5)",None,0,0.0,261831
4,df_itens_pedidos,"(112650, 7)",None,0,0.0,0
5,df_pagamentos,"(103886, 5)",None,0,0.0,0
6,df_pedidos,"(99441, 8)",order_approved_at,160,0.2,0
7,df_pedidos,"(99441, 8)",order_delivered_carrier_date,1783,1.8,0
8,df_pedidos,"(99441, 8)",order_delivered_customer_date,2965,3.0,0
9,df_produtos,"(32951, 9)",product_category_name,610,1.9,0


## 2. Transformação e Limpeza

### Tabela de Avaliações

Antes da aplicação dos modelos de análise de sentimento, os dados textuais passam por um processo de transformação e limpeza com o objetivo de comparar a qualidade das análises realizadas.

Nesta etapa serão realizadas:

- Validação dos tipos de dados
- Unificação dos campos textuais
- Tratamento de valores nulos
- Padronização da escrita
- Remoção de caracteres desnecessários
- Remoção de stopwords

### 2.1 Tipificação dos dados

In [5]:
# STRINGS
dfs['df_avaliacoes']['review_id'] = dfs['df_avaliacoes']['review_id'].astype('string')
dfs['df_avaliacoes']['order_id'] = dfs['df_avaliacoes']['order_id'].astype('string')
dfs['df_avaliacoes']['review_comment_message'] = dfs['df_avaliacoes']['review_comment_message'].astype('string')
dfs['df_avaliacoes']['review_comment_title'] = dfs['df_avaliacoes']['review_comment_title'].astype('string')

# INT
dfs['df_avaliacoes']['review_score'] = dfs['df_avaliacoes']['review_score'].astype('int8') # int8 = economia de memória

# DATA
dfs['df_avaliacoes']['review_creation_date'] = pd.to_datetime(dfs['df_avaliacoes']['review_creation_date'], errors='coerce')
dfs['df_avaliacoes']['review_answer_timestamp'] = pd.to_datetime(dfs['df_avaliacoes']['review_answer_timestamp'], errors='coerce')

dfs['df_avaliacoes'].dtypes

review_id                  string[python]
order_id                   string[python]
review_score                         int8
review_comment_title       string[python]
review_comment_message     string[python]
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

### 2.2 Coluna de texto unificada e dados nulos

**2.2.1 Validando casos onde existe título e não existe comentário**

In [6]:
total = len(dfs['df_avaliacoes'])
count = dfs['df_avaliacoes'][
    dfs['df_avaliacoes']["review_comment_title"].notna() & dfs['df_avaliacoes']["review_comment_message"].isna()
].shape[0]

percent = (count / total) * 100
print(f"{count} linhas ({percent:.2f}%)")

1729 linhas (1.74%)


**2.2.2 Criando coluna unificada com título e comentário**

In [7]:
df_avaliacao = dfs['df_avaliacoes'].copy()

# substitui nulos por string vazia para concatenação
df_avaliacao['review_comment_message'] = df_avaliacao['review_comment_message'].fillna('')
df_avaliacao['review_comment_title'] = df_avaliacao['review_comment_title'].fillna('')

df_avaliacao['review_text'] = (
    df_avaliacao['review_comment_title'] + ' ' + df_avaliacao['review_comment_message']
).str.strip()

# definindo valores vazios para "sem comentários"
df_avaliacao['review_text'] = df_avaliacao['review_text'].replace('', 'sem comentários')

percentual = round((df_avaliacao['review_text'] == 'sem comentários').mean() * 100,2)
print(f"Percentual de valores 'sem comentários' em 'review_text': {percentual}")
df_avaliacao[['review_comment_title', 'review_comment_message', 'review_text']].head(10)

Percentual de valores 'sem comentários' em 'review_text': 56.98


,review_comment_title,review_comment_message,review_text
0,,,sem comentários
1,,,sem comentários
2,,,sem comentários
3,,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado.
4,,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...
5,,,sem comentários
6,,,sem comentários
7,,,sem comentários
8,,,sem comentários
9,recomendo,aparelho eficiente. no site a marca do aparelh...,recomendo aparelho eficiente. no site a marca ...


**2.2.3 Removendo linhas sem comentários**

In [8]:
df_avaliacao = df_avaliacao[
    df_avaliacao['review_text'] != 'sem comentários'
].copy()

print(f"Quantidade de linhas atuais: {df_avaliacao.shape[0]}")

Quantidade de linhas atuais: 42686


### 2.3 Padronizando textos da coluna unificada "review_text"

**2.3.1 Padronização:**
- Tudo minúsculo
- Remoção de acentos e pontuações
- Remoção de espaços extras

In [9]:
import unicodedata
import re

def standardize(text):
  if pd.isnull(text):
    return text

  # minúsculo
  text = text.lower()

  # remove acentos
  text = ''.join(
      c for c in unicodedata.normalize('NFD', text) # NFD: quebra caracteres acentuados em partes separadas (á -> "a" + "´")
      if unicodedata.category(c) != 'Mn' # valida o tipo do caractere (Mn: acentos e diacríticos)
  )

  # remover pontuações
  text = re.sub(r'[^\w\s]', '', text) # substitui tudo que não é letra\número\espaço por ""

  # remover múltiplos espaços
  text = re.sub(r'\s+', ' ', text).strip()

  # garante separação correta de palavras
  palavras = re.findall(r'\b\w+\b', text)
  text = ' '.join(palavras)

  return text

df_avaliacao['review_text_modificado'] = df_avaliacao['review_text'].apply(standardize)

**2.3.2 Removendo stopwords**

_Stopwords_ são palavras comuns que geralmente não carregam muito significado relevante para a análise, como: "o", "a", "de", "da", "para", "com"...

In [10]:
stop_words = set(stopwords.words('portuguese'))

def remover_stopwords(texto):
    if pd.isna(texto):
        return None, []
    tokens = texto.split()
    removidas = [t for t in tokens if t in stop_words]
    tokens = [t for t in tokens if t not in stop_words]
    return (' '.join(tokens) if tokens else None), removidas

resultados = df_avaliacao['review_text_modificado'].apply(remover_stopwords)
df_avaliacao['review_text_modificado'] = resultados.apply(lambda x: x[0])
df_avaliacao['stopwords_removidas'] = resultados.apply(lambda x: x[1])

df_avaliacao = df_avaliacao[df_avaliacao['review_text_modificado'].notna()]

# Verificar exemplo
exemplo = df_avaliacao[df_avaliacao['review_text_modificado'].notna()].iloc[0]
removidas = exemplo['stopwords_removidas']

print(f"Nota: {exemplo['review_score']}")
print(f"Original:  {exemplo['review_text']}")
print(f"Modificado:     {exemplo['review_text_modificado']}")
print(f"\nQuantidade de stopwords removidas: {len(removidas)}")
print(f"Stopwords: {removidas}")

Nota: 5
Original:  Recebi bem antes do prazo estipulado.
Modificado:     recebi bem antes prazo estipulado

Quantidade de stopwords removidas: 1
Stopwords: ['do']


**2.3.3 Removendo colunas que não serão utilizadas**

In [11]:
df_avaliacao = df_avaliacao.drop(columns=['order_id', 'review_creation_date','review_answer_timestamp','review_comment_title', 'review_comment_message', 'review_id'])

## 3. Salvando base final

In [13]:
df_avaliacao.sample(5, random_state=32)

,review_score,review_text,review_text_modificado,stopwords_removidas
59551,1,Esperava um produto original e veio uma réplic...,esperava produto original veio replica segunda...,"[um, e, uma, de]"
3259,5,"Muito bonito A Luminária é muito bonita, atend...",bonito luminaria bonita atendeu expectativa ch...,"[muito, a, e, muito, minha]"
33804,2,"Eu achei a entrega um pouco demorada,e as espe...",achei entrega pouco demoradae especificacoes e...,"[eu, a, um, as, que, estava, no, a, do]"
60651,5,Bom produto,bom produto,[]
47646,1,O jogo de bolas de bilhar que comprei é totalm...,jogo bolas bilhar comprei totalmente diferente...,"[o, de, de, que, e, do, que, foi, a, e, da, ao..."


In [14]:
df_avaliacao.to_parquet("./silver/df_avaliacao.parquet")

## 3. Análise de Sentimento

A análise de sentimento é uma técnica de Processamento de Linguagem Natural (NLP) utilizada para **identificar opiniões, emoções e polaridade presentes em textos**.

Neste projeto, o objetivo é verificar o quanto diferentes modelos conseguem representar corretamente a percepção do consumidor a partir dos comentários realizados nas avaliações.


In [4]:
PARQUET_PATH = "./silver/df_avaliacao.parquet"
df_avaliacao = pd.read_parquet(PARQUET_PATH)

df_avaliacao.sample(5, random_state=32)

,review_score,review_text,review_text_modificado,stopwords_removidas
59551,1,Esperava um produto original e veio uma réplic...,esperava produto original veio replica segunda...,"[um, e, uma, de]"
3259,5,"Muito bonito A Luminária é muito bonita, atend...",bonito luminaria bonita atendeu expectativa ch...,"[muito, a, e, muito, minha]"
33804,2,"Eu achei a entrega um pouco demorada,e as espe...",achei entrega pouco demoradae especificacoes e...,"[eu, a, um, as, que, estava, no, a, do]"
60651,5,Bom produto,bom produto,[]
47646,1,O jogo de bolas de bilhar que comprei é totalm...,jogo bolas bilhar comprei totalmente diferente...,"[o, de, de, que, e, do, que, foi, a, e, da, ao..."


**Função padrão que aplica o modelo de sentimento e retorna um DF padronizado**

In [5]:
def aplicar_modelo_sentimento(df, nome_modelo, func_score, coluna_texto):
    """
    Parâmetros:
    - df: DataFrame original
    - nome_modelo: string com nome do modelo (ex: 'leia')
    - func_score: função que recebe texto e retorna score [-1, 1]
    - coluna_texto: coluna com o texto
    """

    df_result = df.copy()

    # Score
    df_result[f'{nome_modelo}_score'] = df_result[coluna_texto].progress_apply(func_score)

    # Retorno padronizado
    return df_result[[coluna_texto, 'review_score',
                      f'{nome_modelo}_score']]

### 3.1 Vader

O **VADER (Valence Aware Dictionary and sEntiment Reasoner)** é um modelo de análise de sentimento baseado em:

- léxico de palavras
- regras linguísticas
- intensificadores e negações

Originalmente desenvolvido para textos em inglês, o VADER é amplamente utilizado por sua simplicidade e velocidade.

In [18]:
vader = VaderAnalyzer()

def score_vader(text):
  return vader.polarity_scores(text)['compound']

df_vader = aplicar_modelo_sentimento(df_avaliacao, 'vader', score_vader, 'review_text')
df_vader_modificado = aplicar_modelo_sentimento(df_avaliacao, 'vader', score_vader, 'review_text_modificado')

comparacao_vader = pd.DataFrame({
    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    'vader_score_original': df_vader['vader_score'],
    'vader_score_modificado': df_vader_modificado['vader_score']
})

comparacao_vader.sample(10, random_state=32)

100%|██████████| 42558/42558 [00:11<00:00, 3547.40it/s]


,comentario,nota_real,vader_score_original,vader_score_modificado
59551,Esperava um produto original e veio uma réplic...,1,0.3802,0.3182
3259,"Muito bonito A Luminária é muito bonita, atend...",5,0.0000,0.0000
33804,"Eu achei a entrega um pouco demorada,e as espe...",2,-0.2960,0.0000
60651,Bom produto,5,0.0000,0.0000
47646,O jogo de bolas de bilhar que comprei é totalm...,1,0.0000,0.0000
28157,Perfeito tudo de bom Chegou bem antes da data...,4,0.0000,0.0000
65398,Chegou antes do prazo. Tudo OK.,5,0.4466,0.2960
79124,"Produto entregue antes do prazo, muito bem emb...",5,0.0000,0.0000
52014,O material parece de camelo pelo preço não vale,3,0.0000,0.0000
74668,Bom. Faltou a nota fiscal que não chegou com o...,4,-0.5267,0.0000


In [19]:
comparacao_vader.to_parquet("./silver/df_vader.parquet")

### 3.2 LeIA

O **LeIA** é uma adaptação do VADER para língua portuguesa.

Seu funcionamento é semelhante ao VADER, porém utilizando um léxico adaptado para português brasileiro, tornando-o mais adequado para comentários escritos por consumidores brasileiros.

In [20]:
leia = LEIAAnalyzer()

def score_leia(text):
  return leia.polarity_scores(text)['compound']

df_leia = aplicar_modelo_sentimento(df_avaliacao, 'leia', score_leia, 'review_text')
df_leia_modificado = aplicar_modelo_sentimento(df_avaliacao, 'leia', score_leia, 'review_text_modificado')

comparacao_leia = pd.DataFrame({
    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    'leia_score_original': df_leia['leia_score'],
    'leia_score_modificado': df_leia_modificado['leia_score'],
})

comparacao_leia.sample(20, random_state=32)

100%|██████████| 42558/42558 [00:20<00:00, 2042.46it/s]


,comentario,nota_real,leia_score_original,leia_score_modificado
59551,Esperava um produto original e veio uma réplic...,1,0.6360,0.5994
3259,"Muito bonito A Luminária é muito bonita, atend...",5,0.7506,0.7506
33804,"Eu achei a entrega um pouco demorada,e as espe...",2,-0.2960,-0.2960
60651,Bom produto,5,0.4215,0.4215
47646,O jogo de bolas de bilhar que comprei é totalm...,1,0.0000,0.0000
28157,Perfeito tudo de bom Chegou bem antes da data...,4,0.9100,0.9100
65398,Chegou antes do prazo. Tudo OK.,5,0.3885,0.2263
79124,"Produto entregue antes do prazo, muito bem emb...",5,0.5983,0.5574
52014,O material parece de camelo pelo preço não vale,3,-0.2960,-0.2960
74668,Bom. Faltou a nota fiscal que não chegou com o...,4,-0.4215,-0.4215


In [21]:
comparacao_vader.to_parquet("./silver/df_leia.parquet")

### 3.3 Transforms
Os modelos **Transformer** representam o estado da arte em tarefas modernas de Processamento de Linguagem Natural.

Diferentemente dos modelos baseados apenas em palavras isoladas, Transformers conseguem compreender:

- contexto
- relações semânticas
- negação
- intensidade emocional
- dependências entre palavras

Por possuírem maior complexidade computacional, os testes serão realizados inicialmente em uma amostra da base de dados.

**3.3.1 TRANSFORMER 1 — XLM-RoBERTa**

O **XLM-RoBERTa (Robustly Optimized BERT Approach)** é um modelo Transformer multilíngue treinado em diversos idiomas, incluindo português.

O modelo é capaz de compreender contexto textual de forma muito mais profunda que modelos baseados apenas em léxico, permitindo melhor interpretação de:

- frases negativas
- sarcasmo leve
- negações
- contexto semântico

In [ ]:
# !pip install sentencepiece tiktoken protobuf

In [ ]:
# GPU / CPU
device = 0 if torch.cuda.is_available() else -1
print(f'Usando device: {device}')

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


# Pipeline --------------------------------------------------------------
classifier_xlmr = pipeline(
    task='sentiment-analysis',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    tokenizer='cardiffnlp/twitter-xlm-roberta-base-sentiment',
    device=device
)


# Função de conversão para estrelas -------------------------------------
def converter_para_estrelas(label, score):

    label = label.lower()

    # Muito negativo = 1 estrela
    if label == 'negative':

        if score >= 0.75:
            return 1
        else:
            return 2

    # Neutro = 3 estrelas
    elif label == 'neutral':
        return 3

    # Muito positivo = 5 estrelas
    elif label == 'positive':

        if score >= 0.75:
            return 5
        else:
            return 4

    # fallback
    return 3


# Função principal ------------------------------------------------------
def score_xlmr(text):

    try:

        resultado = classifier_xlmr(
            str(text),
            truncation=True,
            max_length=512
        )[0]

        label = resultado['label'].lower()
        score = resultado['score']

        estrelas = converter_para_estrelas(label, score)

        return {
            'label': label,
            'score': score,
            'estrelas': estrelas
        }

    except:

        return 0


# Aplicação do modelo ---------------------------------------------------
tqdm.pandas()

resultado_original = df_avaliacao['review_text'].progress_apply(score_xlmr)
resultado_modificado = df_avaliacao['review_text_modificado'].progress_apply(score_xlmr)


# DataFrames auxiliares -------------------------------------------------
df_xlmr = pd.DataFrame(resultado_original.tolist())
df_xlmr_modificado = pd.DataFrame(resultado_modificado.tolist())


# Comparação ------------------------------------------------------------
comparacao_roberta = pd.DataFrame({

    'comentario': df_avaliacao['review_text'],
    'nota_real': df_avaliacao['review_score'],

    'xlmr_label_original': df_xlmr['label'],
    'xlmr_score_original': df_xlmr['score'],
    'xlmr_estrelas_original': df_xlmr['estrelas'],

    'xlmr_label_modificado': df_xlmr_modificado['label'],
    'xlmr_score_modificado': df_xlmr_modificado['score'],
    'xlmr_estrelas_modificado': df_xlmr_modificado['estrelas']
})

comparacao_roberta.sample(10, random_state=32)

Usando device: -1


 69%|██████▉   | 29430/42558 [1:44:07<13:04, 16.74it/s]      